In [18]:
### heuristictesting.ipynb is a modular and agnostic notebook for testing the M heuristic on different
### horizons.

import pandas as pd
import numpy as np

def m_heuristic_accuracy(df, margin_col='margin', price_open_col='value', price_close_col='value',
                         rows_per_M=12, M_start=1, M_end=10, M_step=0.25):
    """
    Calculate heuristic accuracy across horizons.
    
    Args:
        df : DataFrame with columns for margin, open, close.
        margin_col : str, column name for margin (B - S).
        price_open_col : str, column for horizon open price.
        price_close_col : str, column for horizon close price.
        rows_per_M : int, number of intake rows per M1 (default 12 for S5 -> M1).
        M_start, M_end, M_step : horizon range.

    Returns:
        DataFrame with horizon, rows, acc_pos, acc_neg, acc_avg.
    """
    results = []
    
    def horizon_accuracy(block):
        margin_block = block[margin_col].mean()
        price_change = block[price_close_col].iloc[-1] - block[price_open_col].iloc[0]
        return np.sign(margin_block), np.sign(price_change)

    for m in np.arange(M_start, M_end + M_step, M_step):
        block_size = int(rows_per_M * m)
        
        total_blocks = total_right = 0
        total_p = total_n = 0
        right_p = right_n = 0

        for i in range(0, len(df), block_size):
            block = df.iloc[i:i+block_size]
            if len(block) < block_size:
                continue
            m_sign, p_sign = horizon_accuracy(block)
            
            total_blocks += 1

            if m_sign == p_sign:
                total_right += 1

                if p_sign > 0:
                    total_p += 1
                    right_p += 1
                elif p_sign < 0:
                    total_n += 1
                    right_n += 1
            else:
                if p_sign > 0:
                    total_p += 1
                elif p_sign < 0:
                    total_n += 1

        
        acc = total_right / total_blocks
        acc_p = right_p / total_p
        acc_n = right_n / total_n
        results.append((round(m,2), block_size,total_blocks ,  acc , acc_p , acc_n))
    
    return pd.DataFrame(results, columns=['M','RpH', 'Hcount','Acc' , 'Acc Price Up' , 'Acc Price Down'])

In [20]:
# Recent pandas dataframe.

df = pd.read_csv("full_labeled_USD_JPY_2025-09-22_to_2025-09-25.csv")
accdf = m_heuristic_accuracy(df)
print(m_heuristic_accuracy(df))

        M  RpH  Hcount       Acc  Acc Price Up  Acc Price Down
0    1.00   12    5263  0.486415      0.950020        0.061474
1    1.25   15    4210  0.492162      0.973761        0.035602
2    1.50   18    3509  0.490168      0.981956        0.020677
3    1.75   21    3007  0.490190      0.992497        0.013640
4    2.00   24    2631  0.498670      0.987860        0.008354
5    2.25   27    2339  0.502351      0.995752        0.002828
6    2.50   30    2105  0.511164      0.995353        0.005214
7    2.75   33    1914  0.498955      0.996868        0.000000
8    3.00   36    1754  0.503421      1.000000        0.001233
9    3.25   39    1619  0.513280      1.000000        0.001348
10   3.50   42    1503  0.500998      1.000000        0.000000
11   3.75   45    1403  0.506771      1.000000        0.000000
12   4.00   48    1315  0.515589      1.000000        0.000000
13   4.25   51    1238  0.523425      1.000000        0.000000
14   4.50   54    1169  0.527802      1.000000        0